# RPDA8412w: re-run the visual CNN with pretrained weights (Google Colab)

The reported CNN run was trained from scratch on a CPU and stayed at chance (test ROC-AUC 0.503). This notebook re-runs
**only the CNN and fusion steps** with ImageNet-pretrained weights, larger images and a GPU. The Random Forest and Siamese
results already in the repository are reused as they are.

**Before you start:** menu **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. It takes roughly
30 to 40 minutes. All data are synthetic. At the end a zip file downloads: send it back to Claude to update the README.

## 1. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: choose Runtime > Change runtime type > T4 GPU, then run again"
print("GPU:", torch.cuda.get_device_name(0))

## 2. Get the repository and install requirements

In [ ]:
!git clone -q https://github.com/andynoludwe/ST10505234_RPDA8412w_Fraud_Detection
%cd ST10505234_RPDA8412w_Fraud_Detection
!pip install -q -r requirements.txt

## 3. Render the documents, train and test the CNN, rebuild fusion
`--pretrained` downloads ImageNet weights (works on Colab, not in the offline environment used before). `--image_size 384`
keeps more detail than the earlier 224 pixels. The learning rate defaults to 1e-4 with 2 warm-up epochs when pretrained.

In [ ]:
import time
T0 = time.time()
!python run_all.py --steps docs cnn fusion summary --pretrained --image_size 384 --epochs 40 --patience 8 --batch_size 16

## 4. Result

In [ ]:
import os
for f in ["results/cnn/metrics.json", "results/cnn/predictions.csv", "results/fusion/metrics.json"]:
    assert os.path.getmtime(f) > T0, f + " was NOT updated by this run, so the run failed. Scroll up to the first error and do not use these results."
print(open("results/summary.md").read())
print("".join(open("results/cnn/evaluate_log.txt").readlines()[:12]))

Read the CNN row by **ROC-AUC**, not F1 (75% of the test images are forged, so 'always forged' already gives F1 0.857).
A ROC-AUC clearly above 0.50 means the CNN learned something; the earlier run scored 0.503.

## 5. Download the results

In [ ]:
!zip -q -r cnn_pretrained_results.zip results data_manifests/fusion
from google.colab import files
files.download("cnn_pretrained_results.zip")